# Flag Algebra Method — Worked Examples

This notebook walks through the `FlagAlgebra.jl` package with three progressively harder examples. Each example follows the same four-step recipe:

1. **Specify** the problem as a `FlagProblem`
2. **Generate** all combinatorial data with `build_flag_algebra_data`
3. **Solve** the semidefinite program with `solve_sdp`
4. **Inspect** the certificate and extremal graphs

---

### What is a flag algebra bound?

Suppose you want to know: *how many edges can a triangle-free graph on n vertices have?*  
Mantel's theorem says the answer is $\lfloor n^2/4\rfloor$ — asymptotically, edge density $\leq 1/2$.

Razborov's **flag algebra method** proves such bounds automatically via semidefinite programming.  
The key objects are:

- **Admissible graphs** — the graphs H on n vertices that satisfy all forbidden-subgraph constraints. The SDP produces one scalar λ (the bound) together with PSD matrices Q_σ such that for every admissible H:  
  `density(H) ≤ λ − Σ_σ ⟨Q_σ, P_σ(H)⟩`  
  Since the right-hand side is ≥ density(H) for all H, λ is a valid upper bound on the limiting density.

- **Types** — small labeled subgraphs that anchor the SDP blocks. Each type σ of order s contributes one PSD block Q_σ of size = |flags over σ|.

- **Flags** — extensions of a type: a graph on $m = \lfloor(n+s)/2\rfloor$ vertices whose first $s$ vertices realise the type. The pair density matrix $P_\sigma(H)$ encodes how pairs of flags over $\sigma$ co-embed into $H$.

- **Sharp graphs** — admissible graphs where the bound is achieved with equality (residual = 0). These are the extremal examples.

---

### Setup

If you haven't already installed IJulia (the Julia kernel for Jupyter), run this once:  
```julia
using Pkg; Pkg.add("IJulia")

```
Then launch Jupyter from the `flagmatic_julia` directory so the package environment is active:
```bash

cd /path/to/flagmatic_julia
jupyter notebook --NotebookApp.kernel_name=julia-1.11

```

In [2]:
# Activate the FlagAlgebra package environment.
# If running from the flagmatic_julia directory this picks up Project.toml automatically.
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # adjust path if notebook is moved

using FlagAlgebra
using JuMP
using CSDP         # SDP solver — swap for COSMO, Mosek, etc. if preferred

  Activating project at `~/Desktop/flagmatic_julia`


---
## Example 1 — Mantel's Theorem (K=2, triangle-free graphs)

**Claim:** the maximum edge density of a triangle-free graph is $1/2$.  
**Extremal example:** the complete bipartite graph $K_{n/2, n/2}$.

We take `n=4`, `type_order=2` (the smallest interesting setting). The SDP should return a bound very close to $0.5$.

In [4]:
# Define the forbidden subgraph: triangle K₃ on 3 vertices
k3 = complete_hypergraph(3, Val(2))

# Build the FlagProblem:
#   - K=2 (ordinary graphs), n=4 admissible graph size, type_order=2
#   - forbidden: no K₃ subgraph
#   - minimize=false → maximize edge density
prob1 = FlagProblem(4, 2, Val(2);
                    forbidden=[k3],
                    minimize=false)

# Generate all combinatorial data (types, flags, admissible graphs, pair densities)
data1 = build_flag_algebra_data(prob1)

println("Admissible graphs : ", length(data1.admissible))
println("Types             : ", length(data1.types))
println("Flags per type    : ", map(length, data1.flags))

Admissible graphs : 7
Types             : 3
Flags per type    : [2, 4, 3]


In [5]:
# Solve the SDP
model = CSDP.Optimizer
result1 = solve_sdp(data1, model; extract_Q=true)

println("Status : ", result1.status)
println("Bound  : ", result1.bound, "  (≈ ", round(result1.bound; digits=6), ")")

Status : OPTIMAL
Bound  : 0.500000000677027  (≈ 0.5)


In [51]:
# Verify the rational certificate
cert1 = verify_certificate(data1, result1)

println("Valid certificate : ", cert1.valid)
println("Certified λ      : ", cert1.λ_certified, "  (", float(cert1.λ_certified), ")")
println("Min PSD eigval   : ", round(cert1.min_psd_eigval; digits=8))
println("Min residual     : ", cert1.min_residual)

Valid certificate : false
Certified λ      : 1//2  (0.5)
Min PSD eigval   : 0.0
Min residual     : -2//1563795


In [53]:
# Identify sharp (extremal) graphs — where the bound is tight
sharps1 = identify_sharps(data1, result1)

println("Sharp graph indices : ", sharps1.indices)
println("Sharp graph densities: ", sharps1.densities)
println()
for (g, d) in zip(sharps1.graphs, sharps1.densities)
    println("  density=", d, "  edges=", g.edges)
end

Sharp graph indices : [1]
Sharp graph densities: Rational{Int64}[0]

  density=0//1  edges=Tuple{Int64, Int64}[]


**Interpreting the output:**  
The bound certified at $1/2$ matches Mantel's theorem exactly.  
The sharp graph with edge density $1/2$ is  $K_{2,2}$ on $4$ vertices, confirming it as the extremal example.

---
## Example 2 — K₄⁻-free 3-uniform Hypergraphs (r=3)

**Setup:** 3-uniform hypergraphs (every edge is a triple). Forbid K₄⁻, the unique 3-graph on 4 vertices with exactly 3 edges.  

**Expected bound:** the SDP at `n=5` gives $\approx 1/3$. (The true Turán density is smaller — around $0.2869$ — but `n=5` gives a clean demonstration of the r=3 pipeline.)

In [57]:
# K₄⁻: 3-graph on 4 vertices with edges {1,2,3}, {1,2,4}, {1,3,4}
k4minus = k4_minus()

# For r=3, n=5: valid type orders are 1 and 3 (same parity as n=5, ≤ n-2=3)
prob2 = FlagProblem(5, 3, Val(3);
                    forbidden=[k4minus],
                    minimize=false)

data2 = build_flag_algebra_data(prob2)

println("Admissible graphs : ", length(data2.admissible))
println("Types             : ", length(data2.types), "  (at orders ",
        map(t -> t.type_size, data2.types), ")")
println("Flags per type    : ", map(length, data2.flags))

Admissible graphs : 11
Types             : 3  (at orders [1, 3, 3])
Flags per type    : [2, 7, 4]


In [59]:
model = CSDP.Optimizer
result2 = solve_sdp(data2, model; extract_Q=true)

println("Status : ", result2.status)
println("Bound  : ", result2.bound)

Status : OPTIMAL
Bound  : 0.3333333335597106


The status of the solver is OPTIMAL, which means the solver did its job correctly on the problem it was given. This does not necessarily mean it found the optimal bound for the combinatorial question. Here, increasing $n$ and `type_order` would improve the bound.

In [62]:
# List all admissible graphs so we can pick an interesting one to inspect
for (i, (H, d)) in enumerate(zip(data2.admissible, data2.densities))
    println("H$i: $(length(H.edges)) edges, density=$d  edges=$(H.edges)")
end

# H_idx=1 is the empty graph — its pair density matrices are trivially all-zero
# (no edges means no flag ever appears) or a single 1 on the diagonal
# (the empty flag always pairs with itself). Use the densest admissible graph instead.
H_idx = length(data2.admissible)
H_show = data2.admissible[H_idx]
println("\nInspecting H$H_idx: $(length(H_show.edges)) edges, "
        * "density=$(data2.densities[H_idx]), edges=$(H_show.edges)\n")

# pair_dens[H_idx][σ] is upper-triangular Rational, size nf × nf
for σ in eachindex(data2.types)
    P = data2.pair_dens[H_idx][σ]
    nf = length(data2.flags[σ])
    println("Type σ=$σ (order=", data2.types[σ].type_size, "), ",
            nf, " flags: P is $(size(P,1))×$(size(P,2))")
    display(P)
    println()
end

H1: 0 edges, density=0//1  edges=Tuple{Int64, Int64, Int64}[]
H2: 1 edges, density=1//10  edges=[(1, 2, 3)]
H3: 2 edges, density=1//5  edges=[(1, 2, 3), (1, 2, 4)]
H4: 2 edges, density=1//5  edges=[(1, 2, 3), (1, 4, 5)]
H5: 3 edges, density=3//10  edges=[(1, 2, 3), (1, 2, 4), (1, 2, 5)]
H6: 3 edges, density=3//10  edges=[(1, 2, 3), (1, 2, 4), (1, 3, 5)]
H7: 3 edges, density=3//10  edges=[(1, 2, 3), (1, 2, 4), (3, 4, 5)]
H8: 4 edges, density=2//5  edges=[(1, 2, 3), (1, 2, 4), (1, 2, 5), (3, 4, 5)]
H9: 4 edges, density=2//5  edges=[(1, 2, 3), (1, 2, 4), (1, 3, 5), (1, 4, 5)]
H10: 4 edges, density=2//5  edges=[(1, 2, 3), (1, 2, 4), (1, 3, 5), (2, 4, 5)]
H11: 5 edges, density=1//2  edges=[(1, 2, 3), (1, 2, 4), (1, 3, 5), (2, 4, 5), (3, 4, 5)]

Inspecting H11: 5 edges, density=1//2, edges=[(1, 2, 3), (1, 2, 4), (1, 3, 5), (2, 4, 5), (3, 4, 5)]

Type σ=1 (order=1), 2 flags: P is 2×2


2×2 Matrix{Rational{Int64}}:
 1//3  1//6
  0    1//3


Type σ=2 (order=3), 7 flags: P is 7×7


7×7 Matrix{Rational{Int64}}:
 0  0  0  0  0   0      0
 0  0  0  0  0   0      0
 0  0  0  0  0   0      0
 0  0  0  0  0   0      0
 0  0  0  0  0  1//12  1//12
 0  0  0  0  0   0     1//12
 0  0  0  0  0   0      0


Type σ=3 (order=3), 4 flags: P is 4×4


4×4 Matrix{Rational{Int64}}:
 0  0   0      0
 0  0  1//12  1//12
 0  0   0     1//12
 0  0   0      0

**About the pair density matrix P_σ(H):**  
Entry P[i,j] (i ≤ j) is the probability that, when a random copy of the type σ is embedded in H and then two independent random flag-sized subsets of the remaining vertices are chosen, the induced subgraphs are isomorphic to flag i and flag j respectively.  
These matrices are computed exactly as rationals — verified against the original C flagmatic output.

---
## Example 3 — K₅⁽⁴⁾-free 4-uniform Hypergraphs (r=4, new!)

**Setup:** 4-uniform hypergraphs. Forbid K₅⁽⁴⁾, the complete 4-uniform hypergraph on 5 vertices (all C(5,4)=5 edges).  

This is identical in structure to Examples 1 and 2 — only `Val(4)` and `Fourgraph` change.  
For n=6, type_order=4:
- Valid type orders: 0, 2, 4 (even, ≤ 4)
- s=0 → 1 type (trivial); s=2 → 1 type (no 4-edges possible); s=4 → 2 types (empty or K₄⁽⁴⁾)
- Flag size at s=4: m=(6+4)/2=5 with 1 unlabeled vertex → up to 16 flags per type

**Practical guidance for larger K=4 problems:**  
| n | type_order | ~admissible | ~max flags | expected time |
|---|------------|-------------|------------|---------------|
| 6 | 4 | ~25–120 | 16 | seconds |
| 7 | 4 | ~100–300 | ~100 | minutes–hours |
| 8 | **6** | ~1000+ | 64 | hours–days |
| 8 | 4 | ~1000+ | ~4000 | **impractical** |

For n=8, **use type_order=6** rather than 4 to keep flag sets small.

In [65]:
# K₅⁽⁴⁾: complete 4-uniform hypergraph on 5 vertices
k54 = complete_hypergraph(5, Val(4))

prob3 = FlagProblem(6, 4, Val(4);
                    forbidden=[k54],
                    minimize=false)

data3 = build_flag_algebra_data(prob3)

println("Admissible graphs : ", length(data3.admissible))
println("Types             : ", length(data3.types))
println("Flags per type    : ", map(length, data3.flags))

Admissible graphs : 122
Types             : 4
Flags per type    : [1, 2, 16, 15]


In [67]:
result3 = solve_sdp(data3, CSDP.Optimizer; extract_Q=true)

println("Status : ", result3.status)
println("Bound  : ", result3.bound, "  (≈ ", rationalize(result3.bound; tol=1e-4), ")")

Status : OPTIMAL
Bound  : 0.7500000001538499  (≈ 3//4)


In [69]:
cert3 = verify_certificate(data3, result3)
println("Min PSD eigval : ", round(cert3.min_psd_eigval; digits=8))
println("Min residual   : ", float(cert3.min_residual))
println("λ certified    : ", cert3.λ_certified)

Min PSD eigval : 0.0
Min residual   : 0.0
λ certified    : 3//4


---
## Example 4 — Optimizing an Induced Subgraph Density

Flag algebras also handle **induced density** problems: maximize the density of a specific induced subgraph, rather than edge density.

**Example:** maximize the induced $C_5$-density in triangle-free graphs.  
The exact answer is 24/625, achieved by the $C_5$ blow-up construction.

Notice below that we define $C_5$ by giving the number of vertices, then the set of edges, without using one of our pre-defined helper functions.

In [77]:
# C₅: 5-cycle  (vertices 1-2-4-5-3-1 in this edge encoding)
c5 = Graph(5, [(1,2),(1,3),(2,4),(3,5),(4,5)])
k3 = complete_hypergraph(3, Val(2))

# target=c5 tells the solver to maximize induced density of C₅ (not edge density)
prob4 = FlagProblem(5, 3, Val(2);
                    forbidden=[k3],
                    target=c5,
                    minimize=false)

data4 = build_flag_algebra_data(prob4)
result4 = solve_sdp(data4, CSDP.Optimizer)

println("Bound  : ", result4.bound, "  ≈ ", round(result4.bound; digits=6))
println("Exact  : 24/625 = ", 24/625)

Bound  : 0.03840000190959297  ≈ 0.0384
Exact  : 24/625 = 0.0384


---
## Utilities and Inspection

Useful one-liners for exploring the generated data.

In [80]:
# List all admissible graphs with their edge densities
for (i, (H, d)) in enumerate(zip(data1.admissible, data1.densities))
    println("H$i: ", length(H.edges), " edges, density=", d, "  edges=", H.edges)
end

H1: 0 edges, density=0//1  edges=Tuple{Int64, Int64}[]
H2: 1 edges, density=1//6  edges=[(1, 2)]
H3: 2 edges, density=1//3  edges=[(1, 2), (1, 3)]
H4: 2 edges, density=1//3  edges=[(1, 2), (3, 4)]
H5: 3 edges, density=1//2  edges=[(1, 2), (1, 3), (1, 4)]
H6: 3 edges, density=1//2  edges=[(1, 2), (1, 3), (2, 4)]
H7: 4 edges, density=2//3  edges=[(1, 2), (1, 3), (2, 4), (3, 4)]


In [82]:
# Inspect the Q certificate matrices (one per type)
# A larger eigenvalue gap means a stronger certificate
using LinearAlgebra
for (σ, Q) in enumerate(result1.Q)
    λmin = minimum(eigvals(Symmetric(Q)))
    println("Q_σ=$σ: size=$(size(Q,1))×$(size(Q,2)),  min_eigval=$(round(λmin; digits=8))")
end

Q_σ=1: size=2×2,  min_eigval=0.0
Q_σ=2: size=4×4,  min_eigval=0.0
Q_σ=3: size=3×3,  min_eigval=0.0


In [84]:
# Check subgraph containment manually
path = Graph(3, [(1,2),(2,3)])
triangle = complete_hypergraph(3, Val(2))
println("Path contains triangle?  ", has_subgraph(path, triangle))
println("Triangle contains path?  ", has_subgraph(triangle, path))

# Compute induced density of a specific subgraph
k4 = complete_hypergraph(4, Val(2))
println("Induced path density in K₄: ", induced_density(k4, path))

Path contains triangle?  false
Triangle contains path?  true
Induced path density in K₄: 0//1


In [86]:
# Use multiple threads for faster pair density computation on larger problems.
# Set JULIA_NUM_THREADS before launching Julia, e.g.:
#   julia --threads auto
# or in the shell before starting Jupyter:
#   export JULIA_NUM_THREADS=auto
println("Threads available: ", Threads.nthreads())

Threads available: 1


---
## Quick Reference

### Constructors
```julia
Graph(n, edges)       # r=2 (ordinary graph)
Threegraph(n, edges)  # r=3 (3-uniform hypergraph)
Fourgraph(n, edges)   # r=4 (4-uniform hypergraph)
Hypergraph{K}(n, false, edges)  # general r, not oriented
```

### FlagProblem keyword arguments
```julia
FlagProblem(n, type_order, Val(K);
    forbidden         = [...],   # forbid as subgraph
    forbidden_induced = [...],   # forbid as induced subgraph
    target            = H,       # optimize density of this graph (default: edge density)
    minimize          = false)   # false = maximize (default), true = minimize
```

### Type order rules
- Valid orders s: `n%2, n%2+2, ..., type_order` (must have n−s even)
- Larger type_order → tighter bound but more flags and slower SDP
- For K=4, n=8: **use type_order=6** to avoid very large flag sets

### Full pipeline
```julia
data   = build_flag_algebra_data(prob)          # generate types, flags, pair densities
result = solve_sdp(data, COSMO.Optimizer;       # solve SDP
                   extract_Q=true)              # set true to enable certificate checks
cert   = verify_certificate(data, result)       # exact rational certificate
sharps = identify_sharps(data, result)          # extremal graphs
```